In [ ]:
from google.colab import userdata
import anthropic

api_key = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=20,
    messages=[{"role": "user", "content": "Say OK"}]
)
print(resp.content[0].text)

SecretNotFoundError: Secret ANTHROPIC_API_KEY does not exist.

In [ ]:
from google.colab import userdata
import anthropic

api_key = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=20,
    messages=[{"role": "user", "content": "Say OK"}]
)
print(resp.content[0].text)

SecretNotFoundError: Secret ANTHROPIC_API_KEY does not exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/adtc-msme-project'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project folder ready at: {PROJECT_DIR}")
print("Contents:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
Project folder ready at: /content/drive/MyDrive/adtc-msme-project
Contents: []


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU name: Tesla T4


In [ ]:
import os
print(os.listdir(PROJECT_DIR))

[]


In [ ]:
import os, shutil

MYDRIVE = '/content/drive/MyDrive'
SRC = f'{MYDRIVE}/training_data_v3_final.jsonl'
DST = f'{PROJECT_DIR}/training_data_v3_final.jsonl'

if os.path.exists(SRC):
    shutil.move(SRC, DST)
    print("Moved into project folder.")
elif os.path.exists(DST):
    print("Already in project folder.")
else:
    print("Not found at either location — check the exact filename in Drive.")

print(os.listdir(PROJECT_DIR))

Moved into project folder.
['training_data_v3_final.jsonl']


In [ ]:
broader_signals = [
    "cannot generate", "cannot process this request", "not readable",
    "unreadable", "corrupted", "does not contain any discernible",
    "please provide a properly formatted", "i understand. to help you create",
    "provide a clear, legible", "provide a clear, readable",
    "i can't generate", "i can't provide", "i can't complete",
    "i cannot generate", "i cannot provide", "i cannot complete",
    "doesn't contain regulatory", "doesn't contain information about",
    "doesn't contain actionable", "doesn't contain substantive",
    "no bearing on your", "this text doesn't contain",
    "the source material i have on hand", "please provide the full",
    "please supply appropriate", "i appreciate the question, but the source"
]

corrupted_indices = []
for idx, record in enumerate(all_records):
    for m in record['messages']:
        content_lower = m['content'].lower()
        if any(phrase in content_lower for phrase in broader_signals):
            corrupted_indices.append(idx)
            break

print(f"Found {len(corrupted_indices)} corrupted records")

CLEAN_SRC = f'{PROJECT_DIR}/training_data_v3_clean.jsonl'
corrupted_set = set(corrupted_indices)

with open(CLEAN_SRC, 'w') as f:
    kept = 0
    for idx, record in enumerate(all_records):
        if idx not in corrupted_set:
            f.write(json.dumps(record) + '\n')
            kept += 1

print(f"Kept {kept} clean records, saved to {CLEAN_SRC}")

Found 105 corrupted records
Kept 3308 clean records, saved to /content/drive/MyDrive/adtc-msme-project/training_data_v3_clean.jsonl


In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import anthropic

api_key = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=20,
    messages=[{"role": "user", "content": "Say OK"}]
)
print(resp.content[0].text)

OK


In [ ]:
import json
import re
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

SRC = f'{PROJECT_DIR}/training_data_v3_clean.jsonl'
OUT = f'{PROJECT_DIR}/training_data_v3_reformatted.jsonl'
CHECKPOINT = f'{PROJECT_DIR}/reformat_progress.json'

bold_pattern = re.compile(r'\*\*[^*]+\*\*')
bullet_pattern = re.compile(r'(^|\n)\s*[-*•]\s', re.MULTILINE)
numbered_pattern = re.compile(r'(^|\n)\s*\d+\.\s', re.MULTILINE)
header_pattern = re.compile(r'(^|\n)#{1,4}\s', re.MULTILINE)

def has_structure(text):
    return bool(bold_pattern.search(text) or bullet_pattern.search(text)
                or numbered_pattern.search(text) or header_pattern.search(text))

REWRITE_SYSTEM_PROMPT = """You reformat advisory answers for a Kenyan MSME chatbot to be more readable, without changing any facts, figures, or meaning.

Rules:
- Add structure ONLY where it genuinely helps readability: bold for key terms, deadlines, amounts, and requirement names; bullet lists when there are 2+ distinct items, steps, or requirements; a short numbered list for sequential steps.
- Do NOT add headers (#, ##) — these are short chat answers, not documents.
- Do NOT add structure to answers that are already a single simple point — leave short, single-idea answers as plain sentences.
- Do NOT change, add, or remove any factual content, numbers, agency names, or claims. This is a rewording/formatting pass only.
- Do NOT add a preamble like "Here's the reformatted answer:" — output ONLY the reformatted answer text itself.
- Keep the tone and voice consistent with the original.
- If the original answer is already well-structured or doesn't need any changes, return it unchanged."""

def reformat_one(idx, record):
    messages = record['messages']
    user_msg = next((m['content'] for m in messages if m['role'] == 'user'), '')
    assistant_msgs = [m for m in messages if m['role'] == 'assistant']
    if len(assistant_msgs) != 1:
        return idx, record, False
    original_answer = assistant_msgs[0]['content']
    try:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=REWRITE_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": f"User question:\n{user_msg}\n\nOriginal answer to reformat:\n{original_answer}"}]
        )
        new_answer = resp.content[0].text.strip()
        new_record = json.loads(json.dumps(record))
        for m in new_record['messages']:
            if m['role'] == 'assistant':
                m['content'] = new_answer
        return idx, new_record, True
    except Exception as e:
        print(f"  [error idx {idx}]: {e}")
        return idx, record, False

with open(SRC) as f:
    all_records = [json.loads(line) for line in f]

done_indices = set()
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        done_indices = set(json.load(f))
    print(f"Resuming — {len(done_indices)} already done")

to_process = []
results = {}
if os.path.exists(OUT):
    with open(OUT) as f:
        for line in f:
            r = json.loads(line)
            results[r['_idx']] = r

for idx, record in enumerate(all_records):
    if idx in done_indices:
        continue
    assistant_msgs = [m['content'] for m in record['messages'] if m['role'] == 'assistant']
    text = ' '.join(assistant_msgs)
    if not has_structure(text):
        to_process.append((idx, record))

print(f"Total: {len(all_records)} | Need reformatting: {len(to_process)}")

BATCH_SAVE_EVERY = 25

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(reformat_one, idx, rec): idx for idx, rec in to_process}
    completed = 0
    for future in as_completed(futures):
        idx, new_record, changed = future.result()
        new_record['_idx'] = idx
        results[idx] = new_record
        done_indices.add(idx)
        completed += 1
        if completed % BATCH_SAVE_EVERY == 0:
            with open(CHECKPOINT, 'w') as f:
                json.dump(list(done_indices), f)
            with open(OUT, 'w') as f:
                for r in results.values():
                    f.write(json.dumps(r) + '\n')
            print(f"  progress: {completed}/{len(to_process)}")

with open(CHECKPOINT, 'w') as f:
    json.dump(list(done_indices), f)
with open(OUT, 'w') as f:
    for r in results.values():
        f.write(json.dumps(r) + '\n')

print(f"Done. {completed} examples reformatted.")

Total: 3308 | Need reformatting: 2787
  progress: 25/2787
  progress: 50/2787
  progress: 75/2787
  progress: 100/2787
  progress: 125/2787
  progress: 150/2787
  progress: 175/2787
  progress: 200/2787
  progress: 225/2787
  progress: 250/2787
  progress: 275/2787
  progress: 300/2787
  progress: 325/2787
  progress: 350/2787
  progress: 375/2787
  progress: 400/2787
  progress: 425/2787
  progress: 450/2787
  progress: 475/2787
  progress: 500/2787
  progress: 525/2787
  progress: 550/2787
  progress: 575/2787
  progress: 600/2787
  progress: 625/2787
  progress: 650/2787
  progress: 675/2787
  progress: 700/2787
  progress: 725/2787
  progress: 750/2787
  progress: 775/2787
  progress: 800/2787
  progress: 825/2787
  progress: 850/2787
  progress: 875/2787
  progress: 900/2787
  progress: 925/2787
  progress: 950/2787
  progress: 975/2787
  progress: 1000/2787
  progress: 1025/2787
  progress: 1050/2787
  progress: 1075/2787
  progress: 1100/2787
  progress: 1125/2787
  progress: 11

In [ ]:
FINAL_OUT = f'{PROJECT_DIR}/training_data_v4_merged.jsonl'

reformatted = {}
with open(f'{PROJECT_DIR}/training_data_v3_reformatted.jsonl') as f:
    for line in f:
        r = json.loads(line)
        idx = r.pop('_idx')
        reformatted[idx] = r

final_records = []
reformatted_count = 0
already_structured_count = 0

for idx, record in enumerate(all_records):
    if idx in reformatted:
        final_records.append(reformatted[idx])
        reformatted_count += 1
    else:
        final_records.append(record)
        already_structured_count += 1

with open(FINAL_OUT, 'w') as f:
    for r in final_records:
        f.write(json.dumps(r) + '\n')

print(f"Final dataset: {len(final_records)} examples")
print(f"  - reformatted: {reformatted_count}")
print(f"  - already structured: {already_structured_count}")
print(f"Saved to: {FINAL_OUT}")

Final dataset: 3308 examples
  - reformatted: 2787
  - already structured: 521
Saved to: /content/drive/MyDrive/adtc-msme-project/training_data_v4_merged.jsonl


In [ ]:
import json

with open(f'{PROJECT_DIR}/training_data_v4_merged.jsonl') as f:
    final_records = [json.loads(line) for line in f]

print(f"Total records: {len(final_records)}")

# Broader corruption scan — same signals as before, run against the FINAL merged set
broader_signals = [
    "cannot generate", "cannot process this request", "not readable",
    "unreadable", "corrupted", "does not contain any discernible",
    "please provide a properly formatted", "i understand. to help you create",
    "provide a clear, legible", "provide a clear, readable",
    "i can't generate", "i can't provide", "i can't complete",
    "i cannot generate", "i cannot provide", "i cannot complete",
    "doesn't contain regulatory", "doesn't contain information about",
    "doesn't contain actionable", "doesn't contain substantive",
    "no bearing on your", "this text doesn't contain",
    "the source material i have on hand", "please provide the full",
    "please supply appropriate", "i appreciate the question, but the source",
    "i can't reformat", "i cannot reformat", "unable to reformat",
    "here's the reformatted", "here is the reformatted"  # catch any leftover preamble the reformatting step might have added
]

final_corrupted = []
for idx, record in enumerate(final_records):
    for m in record['messages']:
        content_lower = m['content'].lower()
        if any(phrase in content_lower for phrase in broader_signals):
            final_corrupted.append(idx)
            break

print(f"Corrupted/problematic records found: {len(final_corrupted)}")
if final_corrupted:
    print("Indices:", final_corrupted[:30])

# Structure check — confirm reformatting actually stuck
import re
bold_pattern = re.compile(r'\*\*[^*]+\*\*')
bullet_pattern = re.compile(r'(^|\n)\s*[-*•]\s', re.MULTILINE)
numbered_pattern = re.compile(r'(^|\n)\s*\d+\.\s', re.MULTILINE)
header_pattern = re.compile(r'(^|\n)#{1,4}\s', re.MULTILINE)

def has_structure(text):
    return bool(bold_pattern.search(text) or bullet_pattern.search(text)
                or numbered_pattern.search(text) or header_pattern.search(text))

plain_count = 0
for record in final_records:
    assistant_msgs = [m['content'] for m in record['messages'] if m['role'] == 'assistant']
    text = ' '.join(assistant_msgs)
    if not has_structure(text):
        plain_count += 1

print(f"Still plain prose (no structure): {plain_count} ({100*plain_count/len(final_records):.1f}%)")

Total records: 3308
Corrupted/problematic records found: 0
Still plain prose (no structure): 21 (0.6%)


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.2 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()
print("Confirmed adapter dtype:", next(p.dtype for p in model.parameters() if p.requires_grad))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Confirmed adapter dtype: torch.float16


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

DATA_PATH = f'{PROJECT_DIR}/training_data_v4_merged.jsonl'

dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Output directory now lives on Drive — survives any Colab disconnect
OUTPUT_DIR = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-lora'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,  # reduced from 3 — eval_loss plateaued after epoch 2 last time
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,  # avoid AMP/GradScaler dtype crash from before
    bf16=False,
    max_length=1024,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer initialized — output will save directly to Drive")

Generating train split: 0 examples [00:00, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train: 3142 | Eval: 166


Tokenizing train dataset:   0%|          | 0/3142 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3142 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/166 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/166 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/166 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/166 [00:00<?, ? examples/s]

Trainer initialized — output will save directly to Drive


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.523918,1.498682,1.499529,215200.000000,0.622235
100,1.424232,1.420316,1.441948,431746.000000,0.635886
150,1.410673,1.387392,1.398023,647610.000000,0.640038
200,1.332963,1.365325,1.327342,860428.000000,0.643098
250,1.284927,1.359495,1.271969,1072479.000000,0.646151
300,1.237522,1.352553,1.245724,1290124.000000,0.648242
350,1.298454,1.346659,1.258499,1505995.000000,0.649517
394,1.282715,1.346650,1.256146,1694926.000000,0.649503


TrainOutput(global_step=394, training_loss=1.3941499436567277, metrics={'train_runtime': 4437.6384, 'train_samples_per_second': 1.416, 'train_steps_per_second': 0.089, 'total_flos': 1.674087480121344e+16, 'train_loss': 1.3941499436567277, 'epoch': 2.0})

In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/adtc-msme-project'
OUTPUT_DIR = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-lora'
print(os.listdir(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['training_data_v3_final.jsonl', 'training_data_v3_clean.jsonl', 'reformat_progress.json', 'training_data_v3_reformatted.jsonl', 'training_data_v4_merged.jsonl', 'msme-qwen2.5-1.5b-lora']


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map={"": 0},
)

ADAPTER_PATH = f'{OUTPUT_DIR}/checkpoint-394'
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = merged_model.merge_and_unload()

MERGED_PATH = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-merged'
merged_model.save_pretrained(MERGED_PATH)

merge_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
merge_tokenizer.save_pretrained(MERGED_PATH)

print(f"Merged model saved to {MERGED_PATH}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-merged


In [ ]:
FACT_DIGEST = (
    "You are a helpful assistant advising Kenyan MSME operators on tax, registration, "
    "financing, and regulatory compliance.\\n\\n"
    "Always ground your answers in these verified facts when relevant, and do not contradict them:\\n"
    "- NSSF contribution: 6% employee + 6% employer (matched), Tier I up to KES 9,000, Tier II up to KES 108,000\\n"
    "- Annual leave: minimum 21 working days per 12 months of service (Employment Act Section 28)\\n"
    "- Private limited company registration: NO minimum share capital requirement; stamp duty is 1% of nominal share capital\\n"
    "- Youth Enterprise Development Fund (YEDF): eligibility age 18-34; Rausha loan KES 100,000 (group startup); "
    "Inua loan KES 200,000-1,000,000 (expansion); Vuka loan up to KES 5,000,000 at 8% p.a."
)

with open(f"{MERGED_PATH}/chat_template.jinja") as f:
    template = f.read()

original_default = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

count_before = template.count(original_default)
print(f"Found {count_before} occurrence(s) of the default Qwen message")

template = template.replace(original_default, FACT_DIGEST)

old_user_system = "{{- '<|im_start|>system\\n' + messages[0]['content'] + '<|im_end|>\\n' }}"
new_user_system = "{{- '<|im_start|>system\\n' + '" + FACT_DIGEST + "\\n\\n' + messages[0]['content'] + '<|im_end|>\\n' }}"

count_user_system = template.count(old_user_system)
print(f"Found {count_user_system} occurrence(s) of the user-supplied-system-message line")

template = template.replace(old_user_system, new_user_system)

with open(f"{MERGED_PATH}/chat_template.jinja", "w") as f:
    f.write(template)

print("Chat template patched and saved.")

Found 2 occurrence(s) of the default Qwen message
Found 1 occurrence(s) of the user-supplied-system-message line
Chat template patched and saved.


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3656, done.
remote: Counting objects: 100% (3656/3656), done.
remote: Compressing objects: 100% (2924/2924), done.
remote: Total 3656 (delta 671), reused 3082 (delta 646), pack-reused 0 (from 0)
Receiving objects: 100% (3656/3656), 34.90 MiB | 19.74 MiB/s, done.
Resolving deltas: 100% (671/671), done.
Updating files: 100% (3310/3310), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 904.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 62.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 10.7 MB/s et

In [ ]:
!pip install -q "transformers==4.46.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 45.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.9.2 requires transformers>=4.56.2, but you have transformers 4.46.3 which is incompatible.


In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu tf-keras

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tf_keras 2.20.0
Uninstalling tf_keras-2.20.0:
  Successfully uninstalled tf_keras-2.20.0


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_PATH} \
    --outfile {PROJECT_DIR}/msme-qwen2.5-1.5b-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: msme-qwen2.5-1.5b-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torc

In [ ]:
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release
!cd /content/llama.cpp && cmake --build build --target llama-quantize -j 4

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
    {PROJECT_DIR}/msme-qwen2.5-1.5b-f16.gguf \
    {PROJECT_DIR}/msme-qwen2.5-1.5b-Q4_K_M.gguf \
    Q4_K_M

llama_print_build_info: build = 1 (221f0f6)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf' to '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 28 key-value pairs and 338 tensors from /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:              

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
    {PROJECT_DIR}/msme-qwen2.5-1.5b-f16.gguf \
    {PROJECT_DIR}/msme-qwen2.5-1.5b-Q4_K_M.gguf \
    Q4_K_M

llama_print_build_info: build = 1 (221f0f6)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf' to '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 28 key-value pairs and 338 tensors from /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:              

In [ ]:
import os
path = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-Q4_K_M.gguf'
print(f"Size: {os.path.getsize(path)} bytes")
print(f"Size: {os.path.getsize(path) / (1024*1024):.2f} MB")

Size: 5958752 bytes
Size: 5.68 MB


In [ ]:
# Quantize to LOCAL disk first — reliable, no network sync involved
LOCAL_Q4 = '/content/msme-qwen2.5-1.5b-Q4_K_M.gguf'

!/content/llama.cpp/build/bin/llama-quantize \
    {PROJECT_DIR}/msme-qwen2.5-1.5b-f16.gguf \
    {LOCAL_Q4} \
    Q4_K_M

llama_print_build_info: build = 1 (221f0f6)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf' to '/content/msme-qwen2.5-1.5b-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 28 key-value pairs and 338 tensors from /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f3

In [ ]:
import os
fp16_path = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-f16.gguf'
print(f"fp16 size: {os.path.getsize(fp16_path) / (1024*1024*1024):.2f} GB")

fp16 size: 2.88 GB


In [ ]:
import os

LOCAL_Q4 = '/content/msme-qwen2.5-1.5b-Q4_K_M.gguf'

# Remove any existing (possibly corrupted) file first
if os.path.exists(LOCAL_Q4):
    os.remove(LOCAL_Q4)
    print("Removed old file")

!/content/llama.cpp/build/bin/llama-quantize \
    /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf \
    {LOCAL_Q4} \
    Q4_K_M

# Check size immediately, same cell, no gap
print(f"\nFinal size: {os.path.getsize(LOCAL_Q4) / (1024*1024):.2f} MB")

Removed old file
llama_print_build_info: build = 1 (221f0f6)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf' to '/content/msme-qwen2.5-1.5b-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 28 key-value pairs and 338 tensors from /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general

In [ ]:
from google.colab import files
files.download('/content/msme-qwen2.5-1.5b-Q4_K_M.gguf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

with open(f'{PROJECT_DIR}/training_data_v4_merged.jsonl') as f:
    records = [json.loads(line) for line in f]

print(f"Total records: {len(records)}\n")

# Search for Kajiado anywhere in the conversation
kajiado_matches = []
for idx, record in enumerate(records):
    full_text = ' '.join(m['content'] for m in record['messages'])
    if 'kajiado' in full_text.lower():
        kajiado_matches.append((idx, record))

print(f"Records mentioning 'Kajiado': {len(kajiado_matches)}\n")

for idx, record in kajiado_matches:
    print(f"--- Record {idx} ---")
    for m in record['messages']:
        if m['role'] != 'system':
            print(f"[{m['role']}]: {m['content'][:300]}")
    print()

Total records: 3308

Records mentioning 'Kajiado': 125

--- Record 706 ---
[user]: Create a checklist of social security and corporate responsibility events that my MSME should consider participating in or sponsoring during the remainder of 2023.
[assistant]: Here are the key events you should be aware of for the rest of 2023:

**November Events:**
- **8th–12th November, Mombasa** — Association of Insurance Brokers of Kenya (AIBK) annual regional conference. Consider sponsorship or attendance for networking and insurance knowledge.
- **9th–10th November,

--- Record 707 ---
[user]: What are the key social security or regulatory obligations I should know about as a Kenyan MSME owner?
[assistant]: Based on current regulatory activities, the key focus areas for MSMEs involve **training and professional development opportunities**. There's active engagement from industry bodies like the **Actuarial Society of Kenya** and the **Association of Insurance Brokers** who are running training pro